# 6 Pipeline de Random Forest

In [62]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Laboratorio7_RandomForest")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark listo")

Spark listo


In [63]:
import os

print("Contenido de working_dir:")
print(os.listdir("/opt/app/working_dir/lab7/working_dir"))

Contenido de working_dir:
['constitution.txt', 'lab7', 'nba', 'trump_tweets']


In [64]:
ruta_2025 = "/opt/app/working_dir/lab7/working_dir/lab7/preparado/personas_2025_preparado.parquet"
ruta_2026 = "/opt/app/working_dir/lab7/working_dir/lab7/preparado/personas_2026_preparado.parquet"

df_2025 = spark.read.parquet(ruta_2025)
df_2026 = spark.read.parquet(ruta_2026)

print("2025:" , df_2025.count( ) , "registros" )
print("2026:" , df_2026.count( ) , "registros" )

2025: 53025 registros
2026: 13258 registros


In [65]:
df_2025.select(
    "salario_mensual" ,
    "edad" ,
    
    "antiguedad" ,
    "horas_semanales" ,
    "nivel_educativo" ,
    "categoria_ocupacional" ,
    "dominio"
).show( 10 , truncate = False )

+---------------+----+------------------+---------------+---------------+---------------------+-------+
|salario_mensual|edad|antiguedad        |horas_semanales|nivel_educativo|categoria_ocupacional|dominio|
+---------------+----+------------------+---------------+---------------+---------------------+-------+
|8500.0         |36.0|2.0               |72.0           |3              |2                    |1      |
|3000.0         |47.0|14.0              |60.0           |3              |2                    |2      |
|2000.0         |53.0|15.416666666666666|16.0           |2              |3                    |2      |
|3200.0         |16.0|0.0               |44.0           |3              |2                    |2      |
|3000.0         |20.0|2.4166666666666665|45.0           |4              |2                    |2      |
|1800.0         |53.0|20.5              |45.0           |2              |4                    |2      |
|2300.0         |36.0|6.0               |91.0           |2      

## división de los datos de entrenamiento y validación

In [66]:
train_df, val_df = df_2025.randomSplit([0.8 , 0.2] , seed = 42 )

print( "Entrenamiento:" , train_df.count() )
print("validación:" , val_df.count() )

Entrenamiento: 42570
validación: 10455


## selección de variables predictoras

In [67]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [68]:
categoricas = [ "nivel_educativo", "categoria_ocupacional", "dominio" ]
numericas = [ "edad", "antiguedad",  "horas_semanales" ]

## codificación de variables categóricas con StringIndexer y OneHotEncoder

In [69]:
indexers = [ StringIndexer( inputCol = columna , outputCol = columna + "_index" , handleInvalid = "keep" )
    for columna in categoricas
]

encoder = OneHotEncoder( inputCols = [columna + "_index" for columna in categoricas], outputCols = [columna + "_ohe" for columna in categoricas] )

## construcción del vector de características

In [70]:
assembler = VectorAssembler( inputCols = numericas + [columna + "_ohe" for columna in categoricas] , outputCol = "features" )

## primera configuración de Random Forest

In [71]:
rf_1 = RandomForestRegressor( featuresCol = "features", labelCol = "salario_mensual" , numTrees = 50 , maxDepth = 5 , seed = 42 )
pipeline_rf_1 = Pipeline(stages = indexers + [encoder , assembler, rf_1] )

In [72]:
modelo_rf_1 = pipeline_rf_1.fit(train_df)

Como primera configuración, realizamos un Random Forest con 50 árboles y una profundidad de 5 con una semilla fija de 42 para garantizar que los resultados sean reproducibles. Como Random Forest no necesita estandarización de las variables predictoras, el vector de características se puede usar sin necesidad de cambios.

In [73]:
pred_rf_1 = modelo_rf_1.transform(val_df)

pred_rf_1.select( "salario_mensual", "prediction" ).show(10)

+---------------+------------------+
|salario_mensual|        prediction|
+---------------+------------------+
|         7500.0| 5376.288202436932|
|         3600.0| 4721.207602110189|
|          800.0| 2130.697570032073|
|         3200.0|  2260.63244881584|
|         3500.0| 3888.587929917166|
|         1800.0|1783.1222783650041|
|         2400.0| 2031.631372034763|
|         2800.0|3295.8196098420985|
|         4000.0| 5125.769949017983|
|         3500.0| 3540.403924604453|
+---------------+------------------+
only showing top 10 rows



### Evaluación de la configuración 1

Evaluamos este modelo con MAE, RMSE y R². Para seleccionar la mejor configuración se utiliza principalmente el RMSE, como indica la guía.

In [74]:
evaluador_mae = RegressionEvaluator( labelCol = "salario_mensual" , predictionCol = "prediction", metricName = "mae" )
evaluador_rmse = RegressionEvaluator( labelCol = "salario_mensual" , predictionCol = "prediction" , metricName = "rmse" )
evaluador_r2 = RegressionEvaluator( labelCol ="salario_mensual", predictionCol= "prediction", metricName= "r2" )

mae_1 = evaluador_mae.evaluate(pred_rf_1)
rmse_1 = evaluador_rmse.evaluate(pred_rf_1)
r2_1 = evaluador_r2.evaluate(pred_rf_1)

print("Random Forest de configuración 1")
print("numTrees = 50")
print("maxDepth = 5")
print("MAE:", mae_1)
print("RMSE:", rmse_1)
print("R²:", r2_1)

Random Forest de configuración 1
numTrees = 50
maxDepth = 5
MAE: 1204.2779109312437
RMSE: 2361.6198755462256
R²: 0.3906166170734362


La primera configuración mostró un MAE de Q1,204.28 y un RMSE de Q2,361.62. El R² fue de 0.3906, con esto podemos decir que el modelo explica el 39.1% de la variación en el salario mensual dentro del conjunto de validación.

## Configuración 2

In [75]:
rf_2 = RandomForestRegressor(
    featuresCol= "features",
    labelCol= "salario_mensual",
    numTrees= 100,
    maxDepth= 10 ,
    seed = 42
)

pipeline_rf_2 = Pipeline( stages = indexers + [encoder, assembler, rf_2] ) 

In [76]:
modelo_rf_2 = pipeline_rf_2.fit(train_df )
pred_rf_2 = modelo_rf_2.transform(val_df)

In [77]:
mae_2 = evaluador_mae.evaluate(pred_rf_2)
rmse_2 = evaluador_rmse.evaluate(pred_rf_2)
r2_2 = evaluador_r2.evaluate(pred_rf_2)

print("Random Forest de configuración 2")
print("numTrees = 100")
print("maxDepth = 10")
print("MAE:", mae_2)
print("RMSE:", rmse_2)
print("R²:", r2_2)

Random Forest de configuración 2
numTrees = 100
maxDepth = 10
MAE: 1127.7912525847967
RMSE: 2142.007063705522
R²: 0.49868302047743696


Esta segunda configuración mostró un MAE de Q1,127.79, un RMSE de Q2,142.01 y un R² de 0.4987. En comparación con la primera, esta tiene una disminución tanto en MAE como en RMSE y también hay un aumento en el R². Por ende, hay mejora en la predicción del modelo.

Por ende, el mejor pipeline entrenado es la configuración 2 para Random Forest. Esto ya que tiene el menor RMSE y el mejor R².

| configuración | árboles | profundidad | MAE | RMSE | R² |
|---|---:|---:|---:|---:|---:|
| 1 | 50 | 5 | 1204.28 | 2361.62 | 0.3906 |
| 2 | 100 | 10 | 1127.79 | 2142.01 | 0.4987 |


#### comparación con la regresión lineal

Random Forest y regresión lineal funcionan diferente. La regresión lineal lo que hace es intentar representar la relación entre las variables predictoras y el salario de forma lineal. Por su parte, Random Forest usa árboles de decisión, es decir, como no es lineal representa relaciones más complejas. Por ello, Random Forest puede reflejar mejores valores cuando la relación entre las variables no sigue un comportamiento lineal.


#### mejor modelo (2)

In [83]:
mejor_modelo_rf = modelo_rf_2

In [84]:
ruta_modelo_rf = "/opt/app/working_dir/lab7/working_dir/lab7/modelos/random_forest_mejor"
mejor_modelo_rf.write().overwrite().save(ruta_modelo_rf)

print("Modelo guardado en:")
print(ruta_modelo_rf )

Modelo guardado en:
/opt/app/working_dir/lab7/working_dir/lab7/modelos/random_forest_mejor


In [85]:
import os
print(os.path.exists(ruta_modelo_rf ) )

True


Guardamos el modelo 2.